In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

input_path = Path(
    "/content/drive/MyDrive/Georgetown run/"
    "nearest_intersections_all_locations.csv"
)

print(input_path.exists())
print(input_path)

True
/content/drive/MyDrive/Georgetown run/nearest_intersections_all_locations.csv


In [ ]:
!pip install streetlevel

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.5/327.5 kB 17.4 MB/s eta 0:00:00
  Created wheel for coordinatesconverter: filename=CoordinatesConverter-0.1.5-py3-none-any.whl size=5348 sha256=2c7dbc7bb08f43d5d0bb273c33d5a0b3d65dd4e0dd38dec352e820477967fe5b
  Stored in directory: /root/.cache/pip/wheels/45/28/ab/6dfde78407164f2d7eeb9765daaa2f4a8790293c626006532f
Successfully built coordinatesconverter


In [ ]:
from streetlevel import streetview
print("streetlevel imported successfully")

streetlevel imported successfully


# collection


In [ ]:
# ============================================================
# Download one GSV panorama per nearest-intersection location
#
# Input:
#   /content/drive/MyDrive/Georgetown run/
#       nearest_intersections_all_locations.csv
#
# Images:
#   /content/drive/MyDrive/Georgetown run/
#       intersection imagery/<pumaid>/<point_id>.jpg
#
# Metadata:
#   intersection imagery/
#       intersection_gsv_download_metadata.csv
#       intersection_gsv_download_summary_by_puma.csv
#
# Important:
#   - GSV is searched using intersection_lat/intersection_lon.
#   - Images are named using point_id.
#   - Images are organized into folders using pumaid.
#   - Existing nonempty images are skipped.
# ============================================================

from pathlib import Path
import asyncio
import random
import time

import aiohttp
import pandas as pd
from tqdm.auto import tqdm
from streetlevel import streetview


# ============================================================
# 0. PATHS
# ============================================================

INPUT_CSV = Path(
    "/content/drive/MyDrive/Georgetown run/"
    "nearest_intersections_all_locations.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Georgetown run/"
    "intersection imagery"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_META_CSV = (
    OUTPUT_DIR / "intersection_gsv_download_metadata.csv"
)

SUMMARY_CSV = (
    OUTPUT_DIR / "intersection_gsv_download_summary_by_puma.csv"
)


# ============================================================
# 1. SETTINGS
# ============================================================

# Search radius around the nearest-intersection coordinate.
RADIUS_M = 80

# Panorama download resolution.
ZOOM = 5

# Number of simultaneous downloads.
# Start with 20. Reduce to 10 if many requests fail.
CONCURRENCY = 30

# Total timeout for one HTTP operation.
TIMEOUT_SECONDS = 180

# Number of attempts for each point.
MAX_ATTEMPTS = 3

# Delay between retries.
RETRY_BASE_DELAY_SECONDS = 2

# Number of rows processed before checkpoint metadata is saved.
BATCH_SIZE = 500

# Treat an existing file as valid only when it exceeds this size.
MIN_VALID_IMAGE_BYTES = 1_000


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def clean_id(value):
    """
    Convert an ID to a clean string.

    This prevents integer-like IDs read as floats from becoming
    filenames such as 12345.0.jpg.
    """
    if pd.isna(value):
        return None

    text = str(value).strip()

    if text.endswith(".0"):
        numeric_part = text[:-2]

        if numeric_part.isdigit():
            text = numeric_part

    return text


def valid_existing_image(path):
    """
    Return True only when the image exists and is nonempty.
    """
    return (
        path.exists()
        and path.is_file()
        and path.stat().st_size >= MIN_VALID_IMAGE_BYTES
    )


def get_pano_attribute(pano, possible_names):
    """
    Safely retrieve a panorama attribute when streetlevel versions
    use slightly different attribute names.
    """
    for name in possible_names:
        value = getattr(pano, name, None)

        if value is not None:
            return value

    return None


def normalize_metadata_value(value):
    """
    Convert unusual metadata values into CSV-safe values.
    """
    if value is None:
        return None

    if isinstance(value, (str, int, float, bool)):
        return value

    return str(value)


def save_checkpoint(metadata_rows):
    """
    Save current metadata and a summary by PUMA/status.
    """
    if not metadata_rows:
        return

    metadata = pd.DataFrame(metadata_rows)

    if "point_id" in metadata.columns:
        metadata["point_id"] = metadata["point_id"].astype(str)

        # Keep the most recent result if a point somehow appears twice.
        metadata = metadata.drop_duplicates(
            subset=["point_id"],
            keep="last"
        )

    metadata.to_csv(
        DOWNLOAD_META_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    if {
        "pumaid",
        "download_status"
    }.issubset(metadata.columns):

        summary = (
            metadata
            .groupby(
                ["pumaid", "download_status"],
                dropna=False
            )
            .size()
            .reset_index(name="n")
            .sort_values(
                ["pumaid", "download_status"]
            )
        )

        summary.to_csv(
            SUMMARY_CSV,
            index=False,
            encoding="utf-8-sig"
        )


# ============================================================
# 3. LOAD AND VALIDATE INTERSECTION FILE
# ============================================================

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Input file not found:\n{INPUT_CSV}"
    )

download_frame = pd.read_csv(
    INPUT_CSV,
    dtype={
        "point_id": str,
        "msgid": str,
        "pumaid": str,
    }
)

# Support a file that uses msgid instead of point_id.
if (
    "point_id" not in download_frame.columns
    and "msgid" in download_frame.columns
):
    download_frame = download_frame.rename(
        columns={"msgid": "point_id"}
    )

required_columns = [
    "point_id",
    "pumaid",
    "intersection_lat",
    "intersection_lon",
]

missing_columns = [
    column
    for column in required_columns
    if column not in download_frame.columns
]

if missing_columns:
    raise ValueError(
        "The nearest-intersection file is missing required columns:\n"
        f"{missing_columns}\n\n"
        "Available columns:\n"
        f"{download_frame.columns.tolist()}"
    )

# Clean IDs.
download_frame["point_id"] = (
    download_frame["point_id"]
    .apply(clean_id)
)

download_frame["pumaid"] = (
    download_frame["pumaid"]
    .apply(clean_id)
)

# Convert intersection coordinates to numeric.
download_frame["intersection_lat"] = pd.to_numeric(
    download_frame["intersection_lat"],
    errors="coerce"
)

download_frame["intersection_lon"] = pd.to_numeric(
    download_frame["intersection_lon"],
    errors="coerce"
)

# Identify invalid rows.
invalid_id = (
    download_frame["point_id"].isna()
    | download_frame["pumaid"].isna()
)

invalid_coordinates = (
    download_frame["intersection_lat"].isna()
    | download_frame["intersection_lon"].isna()
    | ~download_frame["intersection_lat"].between(-90, 90)
    | ~download_frame["intersection_lon"].between(-180, 180)
)

invalid_rows = invalid_id | invalid_coordinates

print("=" * 70)
print("INPUT CHECK")
print("=" * 70)
print(f"Input file: {INPUT_CSV}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Total rows: {len(download_frame):,}")
print(
    "Unique point IDs:",
    f"{download_frame['point_id'].nunique(dropna=True):,}"
)
print(
    "Unique PUMAs:",
    f"{download_frame['pumaid'].nunique(dropna=True):,}"
)
print(f"Invalid rows: {int(invalid_rows.sum()):,}")

if download_frame["point_id"].duplicated().any():
    duplicate_ids = (
        download_frame.loc[
            download_frame["point_id"].duplicated(
                keep=False
            ),
            "point_id"
        ]
        .dropna()
        .unique()
    )

    raise ValueError(
        "point_id must be unique, but duplicates were found.\n"
        f"Example duplicate IDs: {duplicate_ids[:10].tolist()}"
    )

if invalid_rows.any():
    print("\nExamples of invalid rows:")

    display(
        download_frame.loc[
            invalid_rows,
            required_columns
        ].head(20)
    )

# Keep valid rows only.
to_download = (
    download_frame.loc[~invalid_rows]
    .copy()
    .reset_index(drop=True)
)

# Create one output folder for each PUMA.
pumaids = sorted(
    to_download["pumaid"].dropna().unique()
)

for pumaid in pumaids:
    (OUTPUT_DIR / pumaid).mkdir(
        parents=True,
        exist_ok=True
    )

print("\nRows by PUMA:")

puma_counts = (
    to_download
    .groupby("pumaid")
    .size()
    .reset_index(name="n_locations")
    .sort_values("pumaid")
)

display(puma_counts)

print("PUMA folders created:", len(pumaids))
print("PUMA IDs:", pumaids)

if len(pumaids) != 9:
    print(
        "\nWARNING: The valid data contain "
        f"{len(pumaids)} PUMAs rather than 9."
    )


# ============================================================
# 4. LOAD PREVIOUS DOWNLOAD METADATA, WHEN AVAILABLE
# ============================================================

previous_metadata = pd.DataFrame()

if DOWNLOAD_META_CSV.exists():
    try:
        previous_metadata = pd.read_csv(
            DOWNLOAD_META_CSV,
            dtype={
                "point_id": str,
                "msgid": str,
                "pumaid": str,
            }
        )

        if (
            "point_id" not in previous_metadata.columns
            and "msgid" in previous_metadata.columns
        ):
            previous_metadata["point_id"] = (
                previous_metadata["msgid"]
            )

        if "point_id" in previous_metadata.columns:
            previous_metadata["point_id"] = (
                previous_metadata["point_id"]
                .apply(clean_id)
            )

            previous_metadata = (
                previous_metadata
                .drop_duplicates(
                    subset=["point_id"],
                    keep="last"
                )
            )

        print(
            "\nPrevious metadata rows loaded:",
            f"{len(previous_metadata):,}"
        )

    except Exception as error:
        print(
            "\nWARNING: Existing metadata could not be read."
        )
        print(
            f"{type(error).__name__}: {error}"
        )

        previous_metadata = pd.DataFrame()


# ============================================================
# 5. DOWNLOAD ONE IMAGE
# ============================================================

async def download_one_gsv(
    session,
    row,
    semaphore
):
    point_id = str(row["point_id"])
    pumaid = str(row["pumaid"])

    intersection_lat = float(
        row["intersection_lat"]
    )

    intersection_lon = float(
        row["intersection_lon"]
    )

    puma_dir = OUTPUT_DIR / pumaid
    puma_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    image_path = (
        puma_dir / f"{point_id}.jpg"
    )

    result = {
        "point_id": point_id,
        "msgid": point_id,
        "pumaid": pumaid,

        # Original sampled location, when present.
        "query_lat": row.get(
            "query_lat",
            None
        ),
        "query_lon": row.get(
            "query_lon",
            None
        ),

        # Intersection used for the GSV search.
        "intersection_lat": intersection_lat,
        "intersection_lon": intersection_lon,

        # Nearest-intersection distance, when present.
        "distance_to_intersection_m": row.get(
            "distance_to_intersection_m",
            row.get(
                "nearest_intersection_distance_m",
                row.get(
                    "distance_m",
                    None
                )
            )
        ),

        "image_path": str(image_path),

        # GSV panorama metadata.
        "gsv_source_id": None,
        "gsv_capture_date": None,
        "gsv_source_lat": None,
        "gsv_source_lon": None,

        "download_status": None,
        "download_error": None,
        "attempts_used": 0,
        "file_size_bytes": None,
    }

    # Resume behavior: skip an already valid image.
    if valid_existing_image(image_path):
        result["download_status"] = (
            "skipped_exists"
        )

        result["file_size_bytes"] = (
            image_path.stat().st_size
        )

        return result

    # Remove a corrupt or empty file before retrying.
    if image_path.exists():
        try:
            image_path.unlink()
        except Exception:
            pass

    async with semaphore:
        # Small stagger prevents all requests from starting
        # at precisely the same instant.
        await asyncio.sleep(
            random.uniform(0.05, 0.20)
        )

        for attempt in range(
            1,
            MAX_ATTEMPTS + 1
        ):
            result["attempts_used"] = attempt

            try:
                pano = (
                    await streetview
                    .find_panorama_async(
                        intersection_lat,
                        intersection_lon,
                        session,
                        radius=RADIUS_M
                    )
                )

                if pano is None:
                    result["download_status"] = (
                        "no_pano_at_intersection"
                    )

                    return result

                result["gsv_source_id"] = (
                    normalize_metadata_value(
                        get_pano_attribute(
                            pano,
                            [
                                "id",
                                "pano_id",
                                "source_id",
                            ]
                        )
                    )
                )

                result["gsv_capture_date"] = (
                    normalize_metadata_value(
                        get_pano_attribute(
                            pano,
                            [
                                "date",
                                "capture_date",
                            ]
                        )
                    )
                )

                result["gsv_source_lat"] = (
                    normalize_metadata_value(
                        get_pano_attribute(
                            pano,
                            [
                                "lat",
                                "latitude",
                            ]
                        )
                    )
                )

                result["gsv_source_lon"] = (
                    normalize_metadata_value(
                        get_pano_attribute(
                            pano,
                            [
                                "lon",
                                "lng",
                                "longitude",
                            ]
                        )
                    )
                )

                await streetview.download_panorama_async(
                    pano,
                    path=str(image_path),
                    session=session,
                    zoom=ZOOM
                )

                if valid_existing_image(image_path):
                    result["download_status"] = "ok"
                    result["download_error"] = None
                    result["file_size_bytes"] = (
                        image_path.stat().st_size
                    )

                    return result

                # Remove an incomplete download.
                if image_path.exists():
                    try:
                        image_path.unlink()
                    except Exception:
                        pass

                raise OSError(
                    "The panorama download completed, "
                    "but the output image is missing "
                    "or too small."
                )

            except Exception as error:
                result["download_error"] = (
                    f"{type(error).__name__}: {error}"
                )

                if image_path.exists():
                    try:
                        if not valid_existing_image(
                            image_path
                        ):
                            image_path.unlink()
                    except Exception:
                        pass

                if attempt < MAX_ATTEMPTS:
                    delay = (
                        RETRY_BASE_DELAY_SECONDS
                        * attempt
                        + random.uniform(0, 1)
                    )

                    await asyncio.sleep(delay)

        result["download_status"] = (
            "error_after_retries"
        )

        return result


# ============================================================
# 6. PROCESS ONE BATCH
# ============================================================

async def run_batch(
    batch_df,
    session,
    semaphore
):
    tasks = [
        asyncio.create_task(
            download_one_gsv(
                session,
                row,
                semaphore
            )
        )
        for _, row in batch_df.iterrows()
    ]

    results = []

    for task in tqdm(
        asyncio.as_completed(tasks),
        total=len(tasks),
        desc="Downloading batch",
        leave=False
    ):
        results.append(await task)

    return results


# ============================================================
# 7. RUN ALL DOWNLOADS WITH CHECKPOINTING
# ============================================================

async def run_all_downloads(
    download_df
):
    timeout = aiohttp.ClientTimeout(
        total=TIMEOUT_SECONDS
    )

    connector = aiohttp.TCPConnector(
        limit=CONCURRENCY,
        limit_per_host=CONCURRENCY
    )

    semaphore = asyncio.Semaphore(
        CONCURRENCY
    )

    # Start from previous metadata so it is preserved.
    metadata_by_point = {}

    if (
        not previous_metadata.empty
        and "point_id"
        in previous_metadata.columns
    ):
        for record in (
            previous_metadata
            .to_dict("records")
        ):
            point_id = clean_id(
                record.get("point_id")
            )

            if point_id is not None:
                metadata_by_point[point_id] = (
                    record
                )

    total_batches = (
        len(download_df)
        + BATCH_SIZE
        - 1
    ) // BATCH_SIZE

    async with aiohttp.ClientSession(
        timeout=timeout,
        connector=connector
    ) as session:

        for batch_number, start_index in enumerate(
            range(
                0,
                len(download_df),
                BATCH_SIZE
            ),
            start=1
        ):
            end_index = min(
                start_index + BATCH_SIZE,
                len(download_df)
            )

            batch_df = download_df.iloc[
                start_index:end_index
            ]

            print(
                f"\nBatch {batch_number}/{total_batches}: "
                f"rows {start_index + 1:,}"
                f"–{end_index:,}"
            )

            batch_results = await run_batch(
                batch_df,
                session,
                semaphore
            )

            for record in batch_results:
                metadata_by_point[
                    str(record["point_id"])
                ] = record

            current_metadata = list(
                metadata_by_point.values()
            )

            save_checkpoint(
                current_metadata
            )

            batch_status = (
                pd.DataFrame(batch_results)
                ["download_status"]
                .value_counts(dropna=False)
            )

            print("Batch result:")
            print(batch_status.to_string())

            print(
                "Checkpoint saved:",
                DOWNLOAD_META_CSV
            )

    final_metadata = pd.DataFrame(
        list(metadata_by_point.values())
    )

    return final_metadata


# ============================================================
# 8. EXECUTE
# ============================================================

print("\n" + "=" * 70)
print("STARTING INTERSECTION GSV DOWNLOAD")
print("=" * 70)
print(f"Search radius: {RADIUS_M} m")
print(f"Zoom: {ZOOM}")
print(f"Concurrency: {CONCURRENCY}")
print(f"Maximum attempts: {MAX_ATTEMPTS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Locations to process: {len(to_download):,}")

start_time = time.time()

download_metadata = await run_all_downloads(
    to_download
)

elapsed_minutes = (
    time.time() - start_time
) / 60

save_checkpoint(
    download_metadata.to_dict("records")
)

print("\n" + "=" * 70)
print("FINAL DOWNLOAD REPORT")
print("=" * 70)
print(
    "Elapsed time:",
    f"{elapsed_minutes:.2f} minutes"
)
print(
    "Metadata rows:",
    f"{len(download_metadata):,}"
)

if not download_metadata.empty:
    print("\nDownload statuses:")

    print(
        download_metadata[
            "download_status"
        ]
        .value_counts(dropna=False)
        .to_string()
    )


# ============================================================
# 9. VERIFY FILE COUNTS BY PUMA
# ============================================================

image_count_rows = []

for pumaid in pumaids:
    puma_dir = OUTPUT_DIR / pumaid

    valid_images = [
        image_path
        for image_path in puma_dir.glob("*.jpg")
        if valid_existing_image(image_path)
    ]

    expected_count = int(
        (
            to_download["pumaid"]
            == pumaid
        ).sum()
    )

    image_count_rows.append(
        {
            "pumaid": pumaid,
            "expected_points": expected_count,
            "valid_image_files": len(
                valid_images
            ),
            "difference": (
                len(valid_images)
                - expected_count
            ),
        }
    )

image_counts = pd.DataFrame(
    image_count_rows
)

print("\nValid image counts by PUMA:")
display(image_counts)

print("\nMetadata saved to:")
print(DOWNLOAD_META_CSV)

print("\nSummary saved to:")
print(SUMMARY_CSV)

print("\nImages saved under:")
print(OUTPUT_DIR)

print("\nExample image location:")
print(
    OUTPUT_DIR
    / "<pumaid>"
    / "<point_id>.jpg"
)

INPUT CHECK
Input file: /content/drive/MyDrive/Georgetown run/nearest_intersections_all_locations.csv
Output directory: /content/drive/MyDrive/Georgetown run/intersection imagery
Total rows: 18,000
Unique point IDs: 18,000
Unique PUMAs: 9
Invalid rows: 0

Rows by PUMA:


,pumaid,n_locations
0,2600100,2000
1,2600802,2000
2,2601200,2000
3,2601600,2000
4,2601701,2000
5,2601703,2000
6,2602903,2000
7,2603203,2000
8,2603212,2000


PUMA folders created: 9
PUMA IDs: ['2600100', '2600802', '2601200', '2601600', '2601701', '2601703', '2602903', '2603203', '2603212']

Previous metadata rows loaded: 11,000

STARTING INTERSECTION GSV DOWNLOAD
Search radius: 80 m
Zoom: 5
Concurrency: 30
Maximum attempts: 3
Batch size: 500
Locations to process: 18,000

Batch 1/36: rows 1–500


Batch result:
download_status
skipped_exists             461
no_pano_at_intersection     39
Checkpoint saved: /content/drive/MyDrive/Georgetown run/intersection imagery/intersection_gsv_download_metadata.csv

Batch 2/36: rows 501–1,000


Batch result:
download_status
skipped_exists             464
no_pano_at_intersection     36
Checkpoint saved: /content/drive/MyDrive/Georgetown run/intersection imagery/intersection_gsv_download_metadata.csv

Batch 3/36: rows 1,001–1,500


...

Batch result:
download_status
ok    500
Checkpoint saved: /content/drive/MyDrive/Georgetown run/intersection imagery/intersection_gsv_download_metadata.csv

Batch 35/36: rows 17,001–17,500


Batch result:
download_status
ok    500
Checkpoint saved: /content/drive/MyDrive/Georgetown run/intersection imagery/intersection_gsv_download_metadata.csv

Batch 36/36: rows 17,501–18,000


Batch result:
download_status
ok    500
Checkpoint saved: /content/drive/MyDrive/Georgetown run/intersection imagery/intersection_gsv_download_metadata.csv

FINAL DOWNLOAD REPORT
Elapsed time: 316.97 minutes
Metadata rows: 18,000

Download statuses:
download_status
skipped_exists             9482
ok                         6964
no_pano_at_intersection    1554

Valid image counts by PUMA:


,pumaid,expected_points,valid_image_files,difference
0,2600100,2000,1847,-153
1,2600802,2000,1789,-211
2,2601200,2000,1588,-412
3,2601600,2000,1602,-398
4,2601701,2000,1651,-349
5,2601703,2000,1993,-7
6,2602903,2000,1976,-24
7,2603203,2000,2000,0
8,2603212,2000,2000,0



Metadata saved to:
/content/drive/MyDrive/Georgetown run/intersection imagery/intersection_gsv_download_metadata.csv

Summary saved to:
/content/drive/MyDrive/Georgetown run/intersection imagery/intersection_gsv_download_summary_by_puma.csv

Images saved under:
/content/drive/MyDrive/Georgetown run/intersection imagery

Example image location:
/content/drive/MyDrive/Georgetown run/intersection imagery/<pumaid>/<point_id>.jpg
